In [1]:
import json
import os
import re
import time
from datetime import datetime

import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
ollama_url = os.getenv("OLLAMA_URL")

print("OpenAI key :", "있음" if openai_api_key else "없음 — .env 를 확인하세요")
print("Ollama URL :", ollama_url)

try:
    probe = OpenAI(api_key="ollama", base_url=ollama_url)
    installed = sorted(m.id for m in probe.models.list())
    print("로컬 모델  :", ", ".join(installed) or "없음")
except Exception as e:
    print("로컬 모델  : 연결 실패 —", type(e).__name__)


python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 3


OpenAI key : 있음
Ollama URL : http://localhost:11434/v1
로컬 모델  : llama3.2:latest, qwen2.5:3b


In [2]:
advisories = json.loads(open("advisories_sample.json", encoding="utf-8").read())

print(f"공지 {len(advisories)}건\n")
for a in advisories:
    print(f"{a['no']:2}. {a['cve_id']:16} {a['label']:22} {len(a['advisory']):5}자")


공지 10건

 1. CVE-2021-44228   Apache Log4j2           2500자
 2. CVE-2022-22965   Spring Framework        1830자
 3. CVE-2017-5638    Apache Struts 2         1726자
 4. CVE-2014-6271    GNU Bash                1011자
 5. CVE-2023-49105   ownCloud                 383자
 6. CVE-2026-48710   Starlette               2500자
 7. CVE-2026-59822   LiteLLM                 1752자
 8. CVE-2026-75650   Adobe Commerce           350자
 9. CVE-2026-49869   Kestra                   772자
10. CVE-2026-73570   Zimbra Collaboration     418자


In [3]:
SYSTEM_PROMPT = """당신은 보안 공지를 요약하는 분석가입니다.

아래 형식으로 **정확히 네 줄**만 답하세요. 다른 말을 덧붙이지 마세요.

제품: <제품 이름과 영향받는 버전>
영향: <공격자가 무엇을 할 수 있는지 한 문장>
조치: <올려야 할 버전. 공지에 버전이 없으면 정확히 "공지에 없음">
긴급도: <긴급|높음|보통|낮음 중 하나>

규칙
- 공지에 적힌 내용만 쓰세요. 공지에 없는 버전·날짜·수치를 지어내지 마세요.
- 버전이 공지에 없으면 반드시 "공지에 없음"이라고 쓰세요. 추측하지 마세요.
- 한국어로 답하세요."""


def build_messages(advisory_text):
    """공지 하나를 messages 로 만든다."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"다음 보안 공지를 요약하세요.\n\n{advisory_text}"},
    ]


print(build_messages(advisories[4]["advisory"]))


[{'role': 'system', 'content': '당신은 보안 공지를 요약하는 분석가입니다.\n\n아래 형식으로 **정확히 네 줄**만 답하세요. 다른 말을 덧붙이지 마세요.\n\n제품: <제품 이름과 영향받는 버전>\n영향: <공격자가 무엇을 할 수 있는지 한 문장>\n조치: <올려야 할 버전. 공지에 버전이 없으면 정확히 "공지에 없음">\n긴급도: <긴급|높음|보통|낮음 중 하나>\n\n규칙\n- 공지에 적힌 내용만 쓰세요. 공지에 없는 버전·날짜·수치를 지어내지 마세요.\n- 버전이 공지에 없으면 반드시 "공지에 없음"이라고 쓰세요. 추측하지 마세요.\n- 한국어로 답하세요.'}, {'role': 'user', 'content': '다음 보안 공지를 요약하세요.\n\nAn issue was discovered in ownCloud owncloud/core before 10.13.1. An attacker can access, modify, or delete any file without authentication if the username of a victim is known, and the victim has no signing-key configured. This occurs because pre-signed URLs can be accepted even when no signing-key is configured for the owner of the files. The earliest affected version is 10.6.0.'}]


In [4]:
openai_client = OpenAI()
ollama_client = OpenAI(api_key="ollama", base_url=ollama_url)   # 키는 아무 값이나 됩니다

# 이름: (클라이언트, 모델 ID, max_tokens)
MODELS = {
    "gpt-4.1-mini": (openai_client, "gpt-4.1-mini", 300),
    "gpt-4.1-nano": (openai_client, "gpt-4.1-nano", 300),
    "qwen2.5:3b":   (ollama_client, "qwen2.5:3b",   300),
    "llama3.2":     (ollama_client, "llama3.2",     300),
}

print("비교할 모델", len(MODELS), "개")
for name, (client, model_id, mt) in MODELS.items():
    where = "로컬" if client is ollama_client else "클라우드"
    print(f"  {name:16} {where:5} -> {model_id:16} max_tokens={mt}")


비교할 모델 4 개
  gpt-4.1-mini     클라우드  -> gpt-4.1-mini     max_tokens=300
  gpt-4.1-nano     클라우드  -> gpt-4.1-nano     max_tokens=300
  qwen2.5:3b       로컬    -> qwen2.5:3b       max_tokens=300
  llama3.2         로컬    -> llama3.2         max_tokens=300


In [5]:
encoding = tiktoken.get_encoding("o200k_base")   # gpt-4.1 계열 토크나이저


def count_tokens(messages):
    return sum(len(encoding.encode(m["content"])) for m in messages)


total_in = 0
print(f"{'번호':>4} {'CVE':16} {'입력 토큰':>9}")
for a in advisories:
    n = count_tokens(build_messages(a["advisory"]))
    total_in += n
    print(f"{a['no']:>4} {a['cve_id']:16} {n:>9,}")

print(f"\n공지 10건 합계: {total_in:,} 토큰 (모델 1개 기준)")
print(f"모델 {len(MODELS)}개면 약 {total_in * len(MODELS):,} 토큰")


  번호 CVE                  입력 토큰
   1 CVE-2021-44228         789
   2 CVE-2022-22965         633
   3 CVE-2017-5638          573
   4 CVE-2014-6271          370
   5 CVE-2023-49105         274
   6 CVE-2026-48710         672
   7 CVE-2026-59822         560
   8 CVE-2026-75650         250
   9 CVE-2026-49869         350
  10 CVE-2026-73570         269

공지 10건 합계: 4,740 토큰 (모델 1개 기준)
모델 4개면 약 18,960 토큰


In [6]:
# no: (제품 낱말, [조치 낱말 — 하나만 맞으면 정답])
ANSWER_KEY = {
    1:  ("Log4j",          ["2.16.0"]),
    2:  ("Spring",         ["5.3.18", "5.2.20"]),
    3:  ("Struts",         ["2.3.32", "2.5.10.1"]),
    4:  ("Bash",           []),            # 공지에 버전 없음
    5:  ("ownCloud",       ["10.13.1"]),
    6:  ("Starlette",      ["1.0.1"]),
    7:  ("LiteLLM",        ["1.84.0"]),
    8:  ("Adobe Commerce", []),            # 공지에 버전 없음
    9:  ("Kestra",         ["1.0.45", "1.3.21"]),
    10: ("Zimbra",         ["10.1.20"]),
}

REQUIRED_LINES = ["제품:", "영향:", "조치:", "긴급도:"]
NO_VERSION = "공지에 없음"


def has_line(answer, prefix):
    """그 줄이 prefix 로 '시작'하는가."""
    return any(line.strip().startswith(prefix) for line in answer.splitlines())


def get_line(answer, prefix):
    """답변에서 '조치:' 같은 줄 하나를 꺼낸다."""
    for line in answer.splitlines():
        if line.strip().startswith(prefix):
            return line.split(":", 1)[1].strip()
    return ""


def grade(no, answer):
    """형식 · 제품 · 조치 세 가지를 채점한다."""
    product, versions = ANSWER_KEY[no]

    form = all(has_line(answer, p) for p in REQUIRED_LINES)
    prod = product.lower() in answer.lower()

    action = get_line(answer, "조치:")
    if versions:
        act = any(v in action for v in versions)
        made_up = False
    else:
        # 버전이 없는 공지 — "공지에 없음" 이라고 해야 정답 (공백은 양쪽 다 지우고 비교)
        act = NO_VERSION.replace(" ", "") in action.replace(" ", "")
        # 숫자.숫자 가 보이면 지어낸 것
        made_up = bool(re.search(r"\d+\.\d+", action))

    return {"형식": form, "제품": prod, "조치": act, "지어냄": made_up}


# 가짜 답으로 채점 함수 테스트
fake_ok  = "제품: Bash 4.3\n영향: 원격 코드 실행\n조치: 공지에 없음\n긴급도: 긴급"
fake_bad = "# 제품: Bash\n# 영향: 원격 코드 실행\n# 조치: 4.4 이상\n# 긴급도: 긴급"
print("정상 답  :", grade(4, fake_ok))
print("엉터리 답:", grade(4, fake_bad))


정상 답  : {'형식': True, '제품': True, '조치': True, '지어냄': False}
엉터리 답: {'형식': False, '제품': True, '조치': False, '지어냄': False}


In [7]:
def ask(client, model_id, advisory_text, max_tokens):
    """공지 하나를 요약시킨다. 클라우드인지 로컬인지 이 함수는 모른다."""
    started = time.time()
    response = client.chat.completions.create(
        model=model_id,
        messages=build_messages(advisory_text),
        temperature=0,        # 같은 입력에 같은 답 — 비교의 전제
        max_tokens=max_tokens,
    )
    choice = response.choices[0]
    usage = response.usage
    return {
        "answer": choice.message.content or "",
        "finish_reason": choice.finish_reason,
        "in_tokens": usage.prompt_tokens,
        "out_tokens": usage.completion_tokens,
        "seconds": round(time.time() - started, 1),
        "error": None,
    }


# 테스트: qwen2.5:3b 에게 5번 공지(ownCloud) 하나만
client, model_id, mt = MODELS["qwen2.5:3b"]
r = ask(client, model_id, advisories[4]["advisory"], mt)

print(r["answer"])
print()
print("finish_reason:", r["finish_reason"])
print("토큰 입력/출력:", r["in_tokens"], "/", r["out_tokens"])
print("시간:", r["seconds"], "초")
print("채점:", grade(5, r["answer"]))


제품: ownCloud owncloud/core
영향: 악의적인 사용자가 알 수 없는 사용자의 파일을 읽고 수정 및 삭제할 수 있습니다.
조치: 10.13.1
긴급도: 높음

finish_reason: stop
토큰 입력/출력: 305 / 55
시간: 17.0 초
채점: {'형식': True, '제품': True, '조치': True, '지어냄': False}


In [8]:
results = {}      # {모델이름: {공지번호: 결과}}

for name, (client, model_id, max_tokens) in MODELS.items():
    print(f"===== {name} =====")
    results[name] = {}
    for a in advisories:
        try:
            r = ask(client, model_id, a["advisory"], max_tokens)
        except Exception as e:
            r = {"answer": "", "finish_reason": None, "in_tokens": 0, "out_tokens": 0,
                 "seconds": 0, "error": f"{type(e).__name__}: {e}"}
            print(f"  {a['no']:2}. {a['cve_id']:16} [오류] {r['error'][:60]}")
        else:
            g = grade(a["no"], r["answer"])
            mark = "".join("O" if g[k] else "X" for k in ("형식", "제품", "조치"))
            warn = "  <- 지어냄" if g["지어냄"] else ""
            print(f"  {a['no']:2}. {a['cve_id']:16} {mark}  "
                  f"{r['finish_reason']:6} {r['seconds']:5.1f}초{warn}")
        results[name][a["no"]] = r
    print()

print("완료")


===== gpt-4.1-mini =====
   1. CVE-2021-44228   OOO  stop     2.4초
   2. CVE-2022-22965   OOO  stop     1.3초
   3. CVE-2017-5638    OOO  stop     1.5초
   4. CVE-2014-6271    OOO  stop     1.3초
   5. CVE-2023-49105   OOO  stop     1.2초
   6. CVE-2026-48710   OOO  stop     1.2초
   7. CVE-2026-59822   OOO  stop     0.9초
   8. CVE-2026-75650   OOO  stop     0.8초
   9. CVE-2026-49869   OOO  stop     1.3초
  10. CVE-2026-73570   OOO  stop     0.9초

===== gpt-4.1-nano =====
   1. CVE-2021-44228   OOX  stop     0.9초
   2. CVE-2022-22965   OOO  stop     1.0초
   3. CVE-2017-5638    OOX  stop     1.0초
   4. CVE-2014-6271    OOO  stop     0.9초
   5. CVE-2023-49105   OOX  stop     0.9초
   6. CVE-2026-48710   OOO  stop     1.0초
   7. CVE-2026-59822   OOO  stop     1.0초
   8. CVE-2026-75650   OOO  stop     0.7초
   9. CVE-2026-49869   OOO  stop     1.2초
  10. CVE-2026-73570   OOX  stop     0.9초

===== qwen2.5:3b =====
   1. CVE-2021-44228   OOO  stop    11.2초
   2. CVE-2022-22965   OOO  stop    12.4초
 

In [9]:
def summarize(name):
    rows = results[name]
    s = {"형식": 0, "제품": 0, "조치": 0, "지어냄": 0,
         "잘림": 0, "오류": 0, "입력": 0, "출력": 0, "초": 0.0}
    for a in advisories:
        r = rows[a["no"]]
        if r["error"]:
            s["오류"] += 1
            continue
        g = grade(a["no"], r["answer"])
        for k in ("형식", "제품", "조치", "지어냄"):
            s[k] += int(g[k])
        if r["finish_reason"] == "length":
            s["잘림"] += 1
        s["입력"] += r["in_tokens"]
        s["출력"] += r["out_tokens"]
        s["초"] += r["seconds"]
    return s


scores = {name: summarize(name) for name in MODELS}

n = len(advisories)
header = (f"{'모델':20}{'형식':>6}{'제품':>6}{'조치':>6}{'지어냄':>7}"
          f"{'잘림':>6}{'오류':>6}{'입력토큰':>10}{'출력토큰':>10}{'총초':>7}")
print(header)
print("-" * len(header))
for name, s in scores.items():
    print(f"{name:20}{s['형식']:>4}/{n}{s['제품']:>4}/{n}{s['조치']:>4}/{n}"
          f"{s['지어냄']:>7}{s['잘림']:>6}{s['오류']:>6}"
          f"{s['입력']:>10,}{s['출력']:>10,}{s['초']:>7.1f}")


모델                      형식    제품    조치    지어냄    잘림    오류      입력토큰      출력토큰     총초
------------------------------------------------------------------------------------
gpt-4.1-mini          10/10  10/10  10/10      0     0     0     4,850       655   12.8
gpt-4.1-nano          10/10  10/10   6/10      0     0     0     4,850       532    9.5
qwen2.5:3b            10/10  10/10   8/10      0     0     0     5,088       952   78.3
llama3.2               9/10  10/10   4/10      1     0     0     5,015       649   73.3


In [10]:
for no in (4, 8):
    label = next(a["label"] for a in advisories if a["no"] == no)
    print("=" * 72)
    print(f"{no}번 — {label}  (공지에 패치 버전 없음)")
    print("=" * 72)
    for name in MODELS:
        r = results[name][no]
        if r["error"]:
            print(f"  {name:20} [오류]")
            continue
        action = get_line(r["answer"], "조치:")
        g = grade(no, r["answer"])
        verdict = "지어냄 !!" if g["지어냄"] else ("정답" if g["조치"] else "애매")
        print(f"  {name:20} [{verdict:9}] {action[:70]}")
    print()


4번 — GNU Bash  (공지에 패치 버전 없음)
  gpt-4.1-mini         [정답       ] 공지에 없음
  gpt-4.1-nano         [정답       ] 공지에 없음
  qwen2.5:3b           [정답       ] 공지에 없음
  llama3.2             [지어냄 !!   ] GNU Bash 4.4 이상

8번 — Adobe Commerce  (공지에 패치 버전 없음)
  gpt-4.1-mini         [정답       ] 공지에 없음
  gpt-4.1-nano         [정답       ] 공지에 없음
  qwen2.5:3b           [정답       ] 공지에 없음
  llama3.2             [정답       ] 공지에 없음



In [11]:
SYSTEM_NO_GUARD = """당신은 보안 공지를 요약하는 분석가입니다.

아래 형식으로 **정확히 네 줄**만 답하세요. 다른 말을 덧붙이지 마세요.

제품: <제품 이름과 영향받는 버전>
영향: <공격자가 무엇을 할 수 있는지 한 문장>
조치: <올려야 할 버전>
긴급도: <긴급|높음|보통|낮음 중 하나>

한국어로 답하세요."""


def ask_with_system(system_prompt, advisory_text, max_tokens=300):
    """system 만 바꿔서 같은 공지를 다시 물어본다."""
    messages = build_messages(advisory_text)
    messages[0] = {"role": "system", "content": system_prompt}   # system 만 교체
    r = openai_client.chat.completions.create(
        model="gpt-4.1-mini", temperature=0, max_tokens=max_tokens, messages=messages)
    return r.choices[0].message.content or ""


for no in (4, 8):
    a = next(x for x in advisories if x["no"] == no)
    print("=" * 72)
    print(f"{no}번 — {a['label']}  (공지에 패치 버전 없음)")
    for tag, sp in [("지시 있음", SYSTEM_PROMPT), ("지시 없음", SYSTEM_NO_GUARD)]:
        for trial in range(3):
            action = get_line(ask_with_system(sp, a["advisory"]), "조치:")
            made_up = bool(re.search(r"\d+\.\d+", action))
            flag = "지어냄 !!" if made_up else "안 지어냄 "
            print(f"  [{tag}] {trial + 1}회  {flag}  조치: {action[:60]}")
    print()


4번 — GNU Bash  (공지에 패치 버전 없음)
  [지시 있음] 1회  안 지어냄   조치: 공지에 없음
  [지시 있음] 2회  안 지어냄   조치: 공지에 없음
  [지시 있음] 3회  안 지어냄   조치: 공지에 없음
  [지시 없음] 1회  안 지어냄   조치: 최신 버전으로 업데이트 필요
  [지시 없음] 2회  안 지어냄   조치: 최신 버전으로 업데이트 필요
  [지시 없음] 3회  안 지어냄   조치: 최신 버전으로 업데이트 필요

8번 — Adobe Commerce  (공지에 패치 버전 없음)
  [지시 있음] 1회  안 지어냄   조치: 공지에 없음
  [지시 있음] 2회  안 지어냄   조치: 공지에 없음
  [지시 있음] 3회  안 지어냄   조치: 공지에 없음
  [지시 없음] 1회  안 지어냄   조치: 최신 보안 패치 적용 권고
  [지시 없음] 2회  안 지어냄   조치: 최신 보안 패치 적용 권고
  [지시 없음] 3회  안 지어냄   조치: 최신 버전으로 업데이트 권장



In [12]:
def save_markdown(filepath="exercise_result.md"):
    """점수표와 답변 전문을 markdown 으로 저장한다."""
    for need, step in [("results", "전체 실행"), ("scores", "점수표")]:
        if need not in globals():
            raise RuntimeError(f"{need} 가 없습니다. '{step}' 셀을 먼저 실행하세요.")

    n = len(advisories)
    lines = [
        "# 보안 공지 요약 — 모델 비교 결과",
        "",
        f"_실행: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}_",
        f"_공지 {n}건 × 모델 {len(MODELS)}개 = {n * len(MODELS)}회 호출_",
        "",
        "## 점수표",
        "",
        "| 모델 | 형식 | 제품 | 조치 | 지어냄 | 잘림 | 오류 | 입력토큰 | 출력토큰 | 총 초 |",
        "|---|---|---|---|---|---|---|---|---|---|",
    ]
    for name, s in scores.items():
        lines.append(
            f"| {name} | {s['형식']}/{n} | {s['제품']}/{n} | {s['조치']}/{n} | "
            f"**{s['지어냄']}** | {s['잘림']} | {s['오류']} | "
            f"{s['입력']:,} | {s['출력']:,} | {s['초']:.1f} |")

    lines += ["", "## 버전이 공지에 없는 두 건", ""]
    for no in (4, 8):
        label = next(a["label"] for a in advisories if a["no"] == no)
        lines += [f"### {no}번 — {label}", ""]
        for name in MODELS:
            r = results[name][no]
            action = "[오류]" if r["error"] else get_line(r["answer"], "조치:")
            lines.append(f"- **{name}**: {action}")
        lines.append("")

    lines += ["## 모델별 전체 답변", ""]
    for name in MODELS:
        lines += [f"### {name}", ""]
        for a in advisories:
            r = results[name][a["no"]]
            lines += [f"**{a['no']}. {a['cve_id']} ({a['label']})**", ""]
            lines += ["```", r["error"] or r["answer"], "```", ""]

    with open(filepath, "w", encoding="utf-8") as f:   # encoding 필수!
        f.write("\n".join(lines))
    print(f"저장: {filepath}")


save_markdown()


저장: exercise_result.md


## 결론

### 1. 어느 모델을 고르겠는가
조건이 없다면 **gpt-4.1-mini**를 고르겠다. 조치 10/10, 지어냄 0, 형식 10/10으로 유일하게 만점이었고,
10건 처리에 12.8초밖에 걸리지 않았다. 답변을 열어 봐도 영향받는 버전과 조치 버전을 정확히 구분해서 썼다.

### 2. 4번·8번에서 지어낸 모델이 있었는가
**llama3.2가 4번 Bash에서 `GNU Bash 4.4 이상`이라고 지어냈다.** 공지에는 "through 4.3"(4.3까지 취약)만 있고 4.4는 없다.
8번은 네 모델 모두 "공지에 없음"으로 정답이었다. 숫자가 아예 없는 공지에서는 지어낼 재료가 없었던 것으로 보인다.
또 llama3.2는 10번에서 공지에 없는 조치(`zimbra-snmp 패키지 제거`)도 만들어 냈는데, 숫자가 없어서 채점 코드가 잡지 못했다.

system 프롬프트에서 "공지에 없음" 지시를 빼고 gpt-4.1-mini로 3회씩 다시 돌렸더니, 버전을 지어내지는 않았지만
"최신 버전으로 업데이트 필요", "최신 보안 패치 적용 권고"처럼 실행할 수 없는 답을 표현을 바꿔 가며 냈다.
지시가 있을 때만 6회 모두 "공지에 없음"으로 일관되게 답했다.

### 3. 점수가 같다면 무엇으로 고르겠는가
지어냄이 0인 모델 중에서는 **조치 점수 → 시간 → 토큰** 순으로 보겠다.
gpt-4.1-nano는 mini보다 3.3초 빠르고 출력 토큰이 123개 적지만, 조치가 6/10으로 4건을 놓쳤다.
틀린 답을 빨리 받는 것은 장점이 아니다.

### 4. 공지를 외부로 보낼 수 없는 조직이라면
**qwen2.5:3b**를 쓰겠다. 비용이 0원이고 데이터가 PC 밖으로 나가지 않으며, 지어냄도 0이다.
감수해야 하는 손해는 이렇다.
- 조치 점수: 10/10 → **8/10** (3번 Struts, 10번 Zimbra를 놓침. 둘 다 버전을 `제품:` 줄에 쓰고 `조치:`로 옮기지 못함)
- 시간: 12.8초 → **78.3초 (약 6배)**
- 품질: 중국어가 섞이거나(`악意大리아자`) 긴급도에 허용되지 않은 값(`중등급`)을 쓰는 경우가 있음

llama3.2는 같은 로컬 모델이지만 조치 4/10에 지어냄이 1건이라 보안 업무에는 쓸 수 없다.

### 채점의 한계
긴급도 값과 영향 문장의 정확성은 채점하지 않았고, 숫자 없이 지어낸 조치는 잡지 못한다.
점수가 실제 품질보다 후하게 나왔을 수 있다.
